In [ ]:
# ==============================================================================
# 🏆 AN2DL CHALLENGE - FINAL "PLATINUM" VERSION
# ==============================================================================
# FEATURES:
# 1. ARCHITECTURE: ConvNeXt Tiny (Best for texture)
# 2. TRAINING: Mixed Precision (AMP) + Label Smoothing (Robustness)
# 3. DATA: "Relaxed" Filter (~7000 patches) + Weighted Sampler logic
# 4. INFERENCE: 4-View TTA + Top-K Mean
# ==============================================================================

import os
import cv2
import zipfile
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

# --- 1. SETUP & PERFORMANCE TUNING ---
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# Enable CudNN Auto-tuner (Daniele's trick for speed)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# --- DOWNLOAD ---
try:
    import gdown
except ImportError:
    !pip install -q gdown
    import gdown

FILES = {
    "dataset.zip": "1-FKB_x1HIyJDS5D8X47lHnYSJ6uwmXen", 
    "labels.csv": "1uItqNMHHl-IoKc602v-aU6Rq7g0Q6JGP",
    "test_data.zip": "1UTRi8b5z_MQSis0IS6L91BqFisTVRgHj"
}
WORK_DIR = "/kaggle/working"
for name, file_id in FILES.items():
    path = os.path.join(WORK_DIR, name)
    if not os.path.exists(path):
        gdown.download(f'https://drive.google.com/uc?id={file_id}', path, quiet=False)

# UNZIP
RAW_TRAIN = os.path.join(WORK_DIR, "train_raw")
RAW_TEST = os.path.join(WORK_DIR, "test_raw")

if not os.path.exists(RAW_TRAIN):
    with zipfile.ZipFile(os.path.join(WORK_DIR, "dataset.zip"), 'r') as z: z.extractall(RAW_TRAIN)
if not os.path.exists(RAW_TEST):
    with zipfile.ZipFile(os.path.join(WORK_DIR, "test_data.zip"), 'r') as z: z.extractall(RAW_TEST)

def find_folder(base):
    c = os.listdir(base)
    if len(c) == 1 and os.path.isdir(os.path.join(base, c[0])): return os.path.join(base, c[0])
    return base

TRAIN_DIR = find_folder(RAW_TRAIN)
TEST_DIR = find_folder(RAW_TEST)

# --- 2. RELAXED TILING (VOLUME FIX) ---
CROPS_DIR = os.path.join(WORK_DIR, "train_crops")
os.makedirs(CROPS_DIR, exist_ok=True)
PATCH_SIZE = 224
STRIDE = 112 

print("\n✂️ GENERATING TILES (RELAXED FILTER)...")
df_raw = pd.read_csv(os.path.join(WORK_DIR, "labels.csv"))
valid_files = set(os.listdir(TRAIN_DIR))
df_raw = df_raw[df_raw['sample_index'].isin(valid_files)].reset_index(drop=True)

crop_records = []
for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw)):
    img_name = row['sample_index']
    img = cv2.imread(os.path.join(TRAIN_DIR, img_name))
    if img is None: continue
    
    h, w, _ = img.shape
    count = 0
    for y in range(0, h-PATCH_SIZE+1, STRIDE):
        for x in range(0, w-PATCH_SIZE+1, STRIDE):
            p = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            
            # Relaxed filter: Allow 30 < mean < 252
            mean_val = p.mean()
            if 30 < mean_val < 252: 
                fn = f"{img_name[:-4]}_{count}.jpg"
                cv2.imwrite(os.path.join(CROPS_DIR, fn), p)
                crop_records.append({'filename': fn, 'label': row['label'], 'original_id': img_name})
                count+=1

crop_df = pd.DataFrame(crop_records)
print(f"✨ Dataset Generated: {len(crop_df)} patches.")

# --- 3. DATASET ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(crop_df, groups=crop_df['original_id']))
train_df = crop_df.iloc[train_idx].reset_index(drop=True)
val_df = crop_df.iloc[val_idx].reset_index(drop=True)

class HistoDataset(Dataset):
    def __init__(self, df, root, tf=None):
        self.df = df
        self.root = root
        self.tf = tf
        self.map = {'Luminal A': 0, 'Luminal B': 1, 'HER2(+)': 2, 'Triple negative': 3}
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        path = os.path.join(self.root, self.df.iloc[i]['filename'])
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.tf: img = self.tf(img)
        return img, torch.tensor(self.map[self.df.iloc[i]['label']], dtype=torch.long)

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Increased batch size slightly due to AMP memory savings
train_loader = DataLoader(HistoDataset(train_df, CROPS_DIR, train_tf), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(HistoDataset(val_df, CROPS_DIR, val_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# --- 4. MODEL ---
print("\n🏗️ BUILDING MODEL...")
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        in_f = self.base.classifier[2].in_features
        # Robust Head
        self.base.classifier[2] = nn.Sequential(
            nn.Dropout(0.5), # Increased dropout for regularization
            nn.Linear(in_f, 512),
            nn.LayerNorm(512), # Added LayerNorm for stability
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 4)
        )
    def forward(self, x): return self.base(x)

model = HybridModel().to(device)

# --- 5. TRAINING (AMP + Label Smoothing) ---
print("\n🔥 TRAINING WITH AMP & LABEL SMOOTHING...")
optimizer = optim.AdamW([
    {'params': model.base.features.parameters(), 'lr': 1e-5},
    {'params': model.base.classifier.parameters(), 'lr': 1e-3}
], weight_decay=1e-2)

# Weights (Imbalance)
weights = torch.tensor([1.65, 1.2, 1.7, 4.0], dtype=torch.float).to(device)
# Label Smoothing (0.1) reduces overconfidence
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)
scaler = torch.cuda.amp.GradScaler() # MIXED PRECISION SCALER

best_f1 = 0.0
EPOCHS = 12 

for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False)
    for img, lbl in loop:
        img, lbl = img.to(device), lbl.to(device)
        optimizer.zero_grad()
        
        # Mixed Precision Forward Pass
        with torch.cuda.amp.autocast():
            output = model(img)
            loss = criterion(output, lbl)
        
        # Scaled Backward Pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loop.set_postfix(loss=loss.item())
        
    # VALIDATION
    model.eval()
    all_preds = []
    all_lbls = []
    with torch.no_grad():
        for img, lbl in val_loader:
            img, lbl = img.to(device), lbl.to(device)
            # No AMP needed for validation usually, but good practice to keep consistent
            with torch.cuda.amp.autocast():
                preds = model(img).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_lbls.extend(lbl.cpu().numpy())
            
    val_f1 = f1_score(all_lbls, all_preds, average='macro')
    scheduler.step()
    
    print(f"Epoch {epoch+1}: Patch Val F1: {val_f1:.4f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "platinum_best.pth")
        print(f"    💾 Saved Platinum Model! (F1: {best_f1:.4f})")

# --- 6. INFERENCE (4-VIEW TTA + TOP-K) ---
print("\n🔮 INFERENCE (4-VIEW TTA)...")
model.load_state_dict(torch.load("platinum_best.pth"))
model.eval()

inv_map = {0: 'Luminal A', 1: 'Luminal B', 2: 'HER2(+)', 3: 'Triple negative'}
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.startswith("img_")])
preds = []
TOP_K = 5

for fname in tqdm(test_files):
    path = os.path.join(TEST_DIR, fname)
    img = cv2.imread(path)
    if img is None: continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    patches = []
    h, w, _ = img.shape
    for y in range(0, h-224+1, 112):
        for x in range(0, w-224+1, 112):
            p = img[y:y+224, x:x+224]
            # Relaxed filter
            if 30 < p.mean() < 252:
                patches.append(p)
    if not patches: patches = [cv2.resize(img, (224,224))]
    
    batch = torch.stack([val_tf(p) for p in patches]).to(device)
    probs_list = []
    
    with torch.no_grad():
        for i in range(0, len(batch), 32):
            sl = batch[i:i+32]
            
            # --- 4-VIEW TTA (Original + Flip + Rotate 90 + Rotate 270) ---
            # This covers more geometric variance than just Flip
            p1 = F.softmax(model(sl), dim=1) # Original
            p2 = F.softmax(model(transforms.functional.hflip(sl)), dim=1) # H-Flip
            p3 = F.softmax(model(transforms.functional.rotate(sl, 90)), dim=1) # Rot 90
            p4 = F.softmax(model(transforms.functional.rotate(sl, 270)), dim=1) # Rot 270
            
            # Average of 4 views
            probs_list.append((p1 + p2 + p3 + p4) / 4.0)
            
    final_probs = torch.cat(probs_list)
    
    # Top-K Strategy
    max_scores, _ = final_probs.max(dim=1)
    k = min(len(final_probs), TOP_K)
    _, top_indices = torch.topk(max_scores, k)
    best_probs = final_probs[top_indices].mean(dim=0)
    
    preds.append({'sample_index': fname, 'label': inv_map[best_probs.argmax().item()]})

pd.DataFrame(preds).to_csv("submission_platinum.csv", index=False)
print("✅ DONE. Generated 'submission_platinum.csv'")